# PRACTICAL 1: Raw Log Ingestion & Structural Exploration

## Objective
Inspect and explore a large raw web application and network security log file (`cj.log`) containing over 2 million lines. We will extract basic file-level characteristics, compute metrics, and perform heuristic analysis to discover active source IPs and timestamp trends without loading the entire dataset into memory to prevent crashes.

## Methodology
1. **Streaming File Processing**: Open the log file in read-only streaming mode to parse it line-by-line.
2. **Metric Computing**: Gather metadata such as size in MB, total line counts, non-empty lines, and character length distribution.
3. **Regex Heuristics**: Detect IP addresses and standardized timestamps using regular expressions.
4. **Visualization**: Plot the distribution of line lengths (revealing payload sizes) and identify the top 15 most active source IP addresses.

In [ ]:
import os
import sys
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Ensure plots directory exists
os.makedirs("../plots", exist_ok=True)

# Configure matplotlib styles
plt.style.use('seaborn-v0_8')
%matplotlib inline

### Stream Processing and Heuristic Log Exploration
Here we execute the generator-based streaming parser. We compile regex patterns for IPv4 addresses and ISO-like timestamps, processing each line to compute character lengths and update frequency counters.

In [ ]:
file_path = "../logs/cj.log"

if not os.path.exists(file_path):
    print(f"[-] Error: Log file not found at {file_path}")
else:
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"[*] Raw log file detected: {file_path}")
    print(f"[*] File Size: {file_size_mb:.2f} MB")
    
    # Heuristic regex patterns
    ip_pattern = re.compile(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b')
    ts_pattern = re.compile(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}')
    
    line_count = 0
    non_empty_count = 0
    line_lengths = []
    
    ip_counter = Counter()
    ts_counter = Counter()
    
    print("[*] Processing raw log lines in streaming mode...")
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line_count += 1
            stripped = line.strip()
            if stripped:
                non_empty_count += 1
                line_lengths.append(len(stripped))
                
                # Heuristic extraction
                ips = ip_pattern.findall(stripped)
                for ip in ips:
                    ip_counter[ip] += 1
                    
                ts_matches = ts_pattern.findall(stripped)
                for ts in ts_matches:
                    ts_counter[ts] += 1
                    
            if line_count % 500000 == 0:
                print(f"    - Processed {line_count} lines...")
                
    print("[+] Stream processing completed.")

### Metrics Summary & Statistics Table
We present a structured summary of the raw characteristics of the log file using the `tabulate` library.

In [ ]:
line_lengths_series = pd.Series(line_lengths)
stats_summary = line_lengths_series.describe()

metrics_data = [
    ["Total Lines Count", f"{line_count:,}"],
    ["Non-Empty Lines Count", f"{non_empty_count:,}"],
    ["File Size (MB)", f"{file_size_mb:.2f} MB"],
    ["Average Line Length", f"{stats_summary['mean']:.2f}"],
    ["Min Line Length", f"{int(stats_summary['min'])}"],
    ["Max Line Length", f"{int(stats_summary['max'])}"],
    ["Unique IPs Found (Heuristics)", f"{len(ip_counter):,}"],
    ["Unique Timestamps Found", f"{len(ts_counter):,}"]
]

print(tabulate(metrics_data, headers=["Metric", "Value"], tablefmt="fancy_grid"))

### Top 15 Most Active Source IPs
We count the frequency of each extracted IP address to identify potential high-volume attackers or automated scanner nodes.

In [ ]:
top_ips = ip_counter.most_common(15)
print(tabulate(top_ips, headers=["IP Address", "Occurrences"], tablefmt="fancy_grid"))

### Visualizations
1. **Line Length Distribution**: An investigation into the variance of log entry lengths, exposing how normal requests compare to large exploit payloads.
2. **Source IP Activity Chart**: A horizontal bar chart ranking the top 15 source IP addresses.

In [ ]:
# Plot 1: Line length distribution histogram + KDE
plt.figure(figsize=(10, 5))
sns.histplot(line_lengths_series, bins=100, kde=True, color='#1f77b4', edgecolor='black', alpha=0.7)
plt.title("Raw Log Line Length Variance & Distribution", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Line Length (Characters)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Top 15 most active source IP addresses
plt.figure(figsize=(12, 6))
ip_df = pd.DataFrame(top_ips, columns=["IP", "Count"])
sns.barplot(data=ip_df, x="Count", y="IP", hue="IP", palette="viridis", legend=False)
plt.title("Top 15 Most Active Source IP Addresses (Heuristic Count)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Request Count", fontsize=12)
plt.ylabel("Source IP Address", fontsize=12)
plt.tight_layout()
plt.show()